# Projeto: Inserção dos dados na camada silver do banco de dados.

### Descrição Geral
Este documento descreve o processo de inserção e normalização dos dados relacionados ao volume de chuva necessário para gerar alertas de alagamento em capitais brasileiras.  
As informações foram coletadas a partir de pesquisas em canais oficiais e consolidadas com base em validações estruturadas realizadas via **Great Expectations (GX)**, assegurando qualidade e conformidade antes de sua persistência na camada *silver*.

O objetivo deste processo é garantir que os dados críticos sobre volumes pluviométricos estejam prontos para análises e cruzamentos com previsões meteorológicas na camada *gold*, com foco na geração de alertas confiáveis.


---

### Informações Técnicas

- **Fonte de Dados**:     
    CEMADEN (Centro Nacional de Monitoramento e Alertas de Desastres Naturais)
    INMET (Instituto Nacional de Meteorologia)
    ANA (Agência Nacional de Águas e Saneamento Básico)
- **Tecnologias Utilizadas**: 
  - Apache Spark (PySpark)
  - Databricks Notebooks

- **Dados persistidos**:
  - Cidade e Unidade Federativa (UF)
  - Data da carga
  - Volume de chuva critico
  - País
---

### Histórico de Atualizações

| Data       | Responsável      | Versão | Descrição                                |
|------------|------------------|--------|--------------------------------------------|
| 18/05/2025 | Lucas Berta      |  1.0   | Criação da inserção dos dados para camada silver|
| 19/05/2025 | Evelise Dias      |  1.1   | Inclusão de documentação sobre normalizações realizadas|
| 25/05/2025 | Evelise Dias      | 1.2    | Adição do pipeline de importação dos dados previamente validados com GX         |
| 26/05/2025 | Evelise Dias      | 1.3    | Atualização da descrição geral do processo e alinhamento com a esteira atual    |

In [0]:
%run ../utils/utils_run_silver

In [0]:
%run ./queries_silver

In [0]:
pasta_validos = lakehouse_logs_gx + "/bronze/chuvas_criticas/validos/chuvas_criticas"
df = ler_tabelas(pasta_validos, fromTable=False)
df = remover_duplicatas(df, ["data_carga", "municipio", "estado", "volume_critico_chuva", "pais"])


# Normalizações

Foram realizadas as normalizações e transformações necessárias para garantir a consistência, qualidade e integridade dos dados antes da inserção na camada *silver*. As ações seguem os padrões definidos de padronização da esteira de dados.

### Normalizações aplicadas:

- **Renomeação de colunas**  
  As colunas foram renomeadas segundo a convenção, garantindo consistência entre tabelas e conformidade com boas práticas de nomenclatura.

- **Conversão de tipo - Data da Carga (`dt_carga`)**  
  O campo `dt_carga` foi convertido para o tipo `timestamp` no formato padrão `"yyyy-MM-dd HH:mm:ss"`.  
  Isso permite ordenação cronológica e análises temporais corretas.

- **Conversão de tipo - Volume Máximo de Chuva (`vol_max_chuva`)**  
  O campo foi convertido para o tipo `decimal`, garantindo precisão nos valores e eliminando riscos de truncamentos ou arredondamentos incorretos.

- **Eliminação de duplicidade via MERGE**  
  A inserção dos dados na tabela *silver* utilizou lógica de **upsert (merge)** com base em:
  - `nom_municipio`
  - `sig_uf`
  - `pais`
  - `vol_max_chuva`  
  Isso assegura que registros duplicados sejam evitados e que atualizações sejam feitas corretamente.

---
Essas normalizações garantem que os dados estejam prontos para análises confiáveis, visualizações e uso em modelos analíticos ou preditivos.


In [0]:
df = renomear_colunas(df, CONVENCAO_COLUNAS_CHUVAS_CRITICAS)

In [0]:
df = alterar_tipos(df, ["dt_carga"], "timestamp", "yyyy-MM-dd HH:mm:ss")

In [0]:
df = alterar_tipos(df, ["vol_max_chuva"], "decimal")

In [0]:
spark.sql(create_table_chuvas_criticas)

In [0]:
df.write.saveAsTable(caminho_tabela_chuvas_criticas_silver, mode="overwrite")